#Naïve Forecasting for Uber Request Logs


Some forecasting methods are extremely simple and surprisingly effective. Naïve forecast is one of them. To create a naïve forecast for "distance per dollar" (defined as distance_to_travel/monetary_cost), first sum the "distance to travel" and "monetary cost" values monthly. This gives the actual value for the current month. For the forecasted value, use the previous month's value. After obtaining both actual and forecasted values, calculate the root mean squared error (RMSE) using the formula RMSE = sqrt(mean(square(actual - forecast))). Report the RMSE rounded to two decimal places.

In [0]:
%skip
%sql
CREATE TABLE ska_catalog.bronze.uber_request_logs(request_id INT, request_date  TIMESTAMP, request_status VARCHAR(10), distance_to_travel FLOAT, monetary_cost FLOAT, driver_to_client_distance FLOAT);

INSERT INTO ska_catalog.bronze.uber_request_logs VALUES (1,'2020-01-09','success', 70.59, 6.56,14.36), (2,'2020-01-24','success', 93.36, 22.68,19.9), (3,'2020-02-08','fail', 51.24, 11.39,21.32), (4,'2020-02-23','success', 61.58,8.04,44.26), (5,'2020-03-09','success', 25.04,7.19,1.74), (6,'2020-03-24','fail', 45.57, 4.68,24.19), (7,'2020-04-08','success', 24.45,12.69,15.91), (8,'2020-04-23','success', 48.22,11.2,48.82), (9,'2020-05-08','success', 56.63,4.04,16.08), (10,'2020-05-23','fail', 19.03,16.65,11.22), (11,'2020-06-07','fail', 81,6.56,26.6), (12,'2020-06-22','fail', 21.32,8.86,28.57), (13,'2020-07-07','fail', 14.74,17.76,19.33), (14,'2020-07-22','success',66.73,13.68,14.07), (15,'2020-08-06','success',32.98,16.17,25.34), (16,'2020-08-21','success',46.49,1.84,41.9), (17,'2020-09-05','fail', 45.98,12.2,2.46), (18,'2020-09-20','success',3.14,24.8,36.6), (19,'2020-10-05','success',75.33,23.04,29.99), (20,'2020-10-20','success', 53.76,22.94,18.74);


In [0]:
SELECT * FROM ska_catalog.bronze.uber_request_logs;

In [0]:
WITH agg_month AS (
  SELECT
    Date_format(request_date, 'yyyy-MM') AS `year_month`, 
    ROUND(SUM(distance_to_travel),8)as `total_distance`,
    ROUND(SUM(monetary_cost),8) AS `total_cost`
  FROM  ska_catalog.bronze.uber_request_logs
  GROUP BY year_month
),
dist_per_dollar AS (
  SELECT *,
  ROUND((total_distance / total_cost),8) AS dist_per_dollar
  FROM agg_month
),
naive_forecast AS (
  SELECT year_month, dist_per_dollar,
  LAG(dist_per_dollar, 1) OVER (ORDER BY year_month) AS forecasted_value
  FROM dist_per_dollar
)
SELECT *, ROUND(sqrt(AVG(power(dist_per_dollar - forecasted_value,2))),2) AS RSME
FROM naive_forecast
WHERE forecasted_value IS NOT NULL
GROUP BY year_month, dist_per_dollar, forecasted_value

In [0]:
-- How many total requests are in the table?
SELECT COUNT(*) AS request_cnt FROM ska_catalog.bronze.uber_request_logs

In [0]:
-- What is the count of requests by request_status (success vs fail)?
SELECT INITCAP(request_status) AS `STATUS`, COUNT(*) AS request_count
FROM ska_catalog.bronze.uber_request_logs
GROUP BY request_status;

In [0]:
-- What is the average monetary_cost per request?
SELECT ROUND(AVG(monetary_cost), 2) AS avg_monetary_cost_per_request
FROM ska_catalog.bronze.uber_request_logs;

In [0]:
SELECT MONTH(request_date) AS `MONTH`,date_format(request_date,'MMMM') AS `MONTH_NAME`,
ROUND(SUM(distance_to_travel),4) AS  `total_distance`, 
ROUND(SUM(monetary_cost),4) AS `total_cost`
FROM ska_catalog.bronze.uber_request_logs
GROUP BY MONTH , MONTH_NAME
ORDER BY  MONTH , MONTH_NAME

In [0]:
SELECT
request_id AS `ID`,
DATE_FORMAT(request_date, 'yyyy-MM-dd') AS `DATE`,
ROUND(SUM(distance_to_travel) / NULLIF(SUM(monetary_cost), 0),2) AS `DISTANCE PER DOLLAR`
FROM ska_catalog.bronze.uber_request_logs
GROUP BY ID,DATE

In [0]:
-- Which requests have driver_to_client_distance greater than 30 and status = 'success'?

SELECT * FROM ska_catalog.bronze.uber_request_logs
WHERE CAST(driver_to_client_distance AS INT) > 30 AND request_status LIKE 'success'

In [0]:
-- What are the minimum and maximum values of distance_to_travel and monetary_cost?
SELECT ROUND(MAX(monetary_cost),2) AS MAX_monetary_cost,
  ROUND(MIN(monetary_cost),2) AS MIN_monetary_cost,
  ROUND(MAX(distance_to_travel),2) AS MAX_distance_to_travel,
  ROUND(MIN(distance_to_travel),2) AS MIN_distance_to_travel
  FROM ska_catalog.bronze.uber_request_logs

In [0]:
-- What are the top 5 requests with the highest monetary_cost?
SELECT * FROM ska_catalog.bronze.uber_request_logs
ORDER BY monetary_cost DESC
LIMIT 5;

In [0]:
-- Extract year and month from request_date and count requests per month.
SELECT
  DATE_FORMAT(request_date, 'yyyy-MM') AS year_month,
  COUNT(*) AS request_count
FROM ska_catalog.bronze.uber_request_logs
GROUP BY year_month
ORDER BY year_month;

In [0]:
UPDATE ska_catalog.bronze.uber_request_logs
SET monetary_cost = 22.68000030517578 
WHERE request_id = 2;

In [0]:
-- Replace negative monetary_cost values with NULL.
UPDATE ska_catalog.bronze.uber_request_logs
SET monetary_cost = NULL 
WHERE monetary_cost < 0;

In [0]:
SELECT * FROM ska_catalog.bronze.uber_request_logs

In [0]:
-- Compute monthly distance per dollar and apply a naïve forecast using the previous month’s value.

WITH agg_month(
  SELECT date_format(request_date, 'yyyy-MM') AS `year_month`,
  SUM(distance_to_travel) AS `total_distance`,
  SUM(monetary_cost) AS `total_cost`
  FROM ska_catalog.bronze.uber_request_logs
  GROUP BY year_month
),
dist_per_dollar AS (
  SELECT year_month,total_distance,total_cost,
  ROUND ( (total_distance/ IF(total_cost = 0,NULL,total_cost)),2) AS `dist_per_dollar`
  FROM agg_month
)
SELECT year_month,dist_per_dollar,
LAG(dist_per_dollar,1) OVER(ORDER BY year_month) AS `forecasted_cost`
FROM dist_per_dollar;

In [0]:
-- Add a 3-month moving average of distance per dollar.
WITH agg_month AS (
  SELECT date_format(request_date, 'yyyy-MM') AS `year_month`,
    SUM(distance_to_travel) AS `total_distance`,
    SUM(monetary_cost) AS `total_cost`
  FROM ska_catalog.bronze.uber_request_logs
  GROUP BY year_month
),
dist_per_dollar AS (
  SELECT year_month, total_distance, total_cost,
    ROUND((total_distance / NULLIF(total_cost, 0)), 2) AS `dist_per_dollar`
  FROM agg_month
)
SELECT year_month, dist_per_dollar,
  ROUND(AVG(dist_per_dollar) OVER (ORDER BY year_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS `moving_avg`
FROM dist_per_dollar